In [0]:
SELECT COUNT(*) FROM
bootcamp.silver.propiedades;
DESCRIBE TABLE bootcamp.silver.propiedades;

In [0]:
%sql
SELECT
  COUNT(DISTINCT partido) AS dist_partido,
  COUNT(DISTINCT tipo_operacion) AS dist_tipo_op,
  COUNT(DISTINCT estado) AS dist_estado,
  COUNT(DISTINCT moneda) AS dist_moneda,
  COUNT(DISTINCT precio) AS dist_precio,
  COUNT(DISTINCT metros_cuadrados_totales) AS dist_m2,
  COUNT(DISTINCT url) AS dist_url,
  COUNT(*) AS total_filas
FROM bootcamp.silver.propiedades;

In [0]:
%sql
-- Paso 1: Vista staging con correcciones
CREATE OR REPLACE TEMP VIEW staging_correcciones AS
SELECT partido, region, 'Gran Buenos Aires' AS ciudad
FROM bootcamp.gold.dim_zona
WHERE region = 'gba norte' AND ciudad = 'GBA';

In [0]:
%sql
-- Verificar COUNT antes del MERGE
SELECT COUNT(*) AS count_antes FROM bootcamp.gold.dim_zona;

In [0]:
%sql
-- Paso 2: MERGE SCD1 (solo UPDATE, nunca INSERT de segunda version)
MERGE INTO bootcamp.gold.dim_zona AS target
USING staging_correcciones AS source
ON target.partido = source.partido AND target.region = source.region
WHEN MATCHED THEN
  UPDATE SET ciudad = source.ciudad;

In [0]:
%sql
-- Paso 3: Verificar que no se crearon filas nuevas
SELECT COUNT(*) AS count_despues FROM bootcamp.gold.dim_zona;

In [0]:
%sql
-- Paso 1: DDL con columnas de control SCD2
CREATE TABLE IF NOT EXISTS bootcamp.gold.dim_zona_scd2 (
  zona_id       BIGINT GENERATED ALWAYS AS IDENTITY,
  partido       STRING NOT NULL,
  region        STRING NOT NULL,
  ciudad        STRING,
  provincia     STRING DEFAULT 'Buenos Aires',
  pais          STRING DEFAULT 'Argentina',
  valid_from    TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP(),
  valid_to      TIMESTAMP DEFAULT '9999-12-31',
  is_current    BOOLEAN DEFAULT TRUE
) USING DELTA
TBLPROPERTIES('delta.feature.allowColumnDefaults' = 'supported');

In [0]:
%sql
-- Paso 3: Verificar que todos son is_current = TRUE
SELECT
  COUNT(*) AS total,
  SUM(CASE WHEN is_current = TRUE THEN 1 ELSE 0 END) AS actuales,
  SUM(CASE WHEN is_current = FALSE THEN 1 ELSE 0 END) AS historicos
FROM bootcamp.gold.dim_zona_scd2;

In [0]:
%sql
-- Paso 1: CERRAR la version actual de 'capital federal'
-- IMPORTANTE: el BK en Silver es 'capital federal' (con espacio), no 'capital-federal' (guion)
UPDATE bootcamp.gold.dim_zona_scd2
SET valid_to = CURRENT_TIMESTAMP(), is_current = FALSE
WHERE partido = 'capital federal' AND region = 'capital federal' AND is_current = TRUE;

In [0]:
%sql
-- Paso 2: INSERTAR nueva version (mismo BK, distinto valor de ciudad)
INSERT INTO bootcamp.gold.dim_zona_scd2 (partido, region, ciudad, valid_from, valid_to, is_current)
VALUES ('capital federal', 'capital federal', 'Buenos Aires', CURRENT_TIMESTAMP(), '9999-12-31', TRUE);

In [0]:
%sql
-- Paso 3: Verificar - debe mostrar 2 filas (version cerrada + version nueva)
SELECT zona_id, partido, region, ciudad, valid_from, valid_to, is_current
FROM bootcamp.gold.dim_zona_scd2
WHERE partido = 'capital federal' AND region = 'capital federal'
ORDER BY valid_from;

In [0]:
%sql
-- Paso 1: Vista staging con cambios simulados
CREATE OR REPLACE TEMP VIEW staging_cambios_scd2 AS
SELECT * FROM VALUES
  ('palermo', 'capital federal', 'CABA Sur'),
  ('barrio-nuevo', 'capital federal', 'CABA')
AS t(partido, region, ciudad_nueva);

In [0]:
%sql
-- Paso 2: MERGE para CERRAR registros que cambiaron
MERGE INTO bootcamp.gold.dim_zona_scd2 AS target
USING staging_cambios_scd2 AS source
ON target.partido = source.partido
   AND target.region = source.region
   AND target.is_current = TRUE
WHEN MATCHED AND target.ciudad != source.ciudad_nueva THEN
  UPDATE SET valid_to = CURRENT_TIMESTAMP(), is_current = FALSE
WHEN NOT MATCHED THEN
  INSERT (partido, region, ciudad, valid_from, valid_to, is_current)
  VALUES (source.partido, source.region, source.ciudad_nueva, CURRENT_TIMESTAMP(), '9999-12-31', TRUE);

In [0]:
%sql
-- Paso 3: INSERTAR la nueva version de los registros que acabamos de cerrar en el paso anterior
-- (solo si: hay una version cerrada hace < 1 minuto, y NO existe ya una version actual)
INSERT INTO bootcamp.gold.dim_zona_scd2 (partido, region, ciudad, valid_from, valid_to, is_current)
SELECT
  s.partido,
  s.region,
  s.ciudad_nueva,
  CURRENT_TIMESTAMP(),
  '9999-12-31',
  TRUE
FROM staging_cambios_scd2 s
WHERE EXISTS (
    -- el registro fue cerrado recientemente por el MERGE del paso 2
    SELECT 1
    FROM bootcamp.gold.dim_zona_scd2 t
    WHERE t.partido = s.partido
      AND t.region = s.region
      AND t.is_current = FALSE
      AND t.valid_to >= CURRENT_TIMESTAMP() - INTERVAL 1 MINUTE
  )
  AND NOT EXISTS (
    -- evita duplicar si ya existe una version actual para ese BK
    SELECT 1
    FROM bootcamp.gold.dim_zona_scd2 t2
    WHERE t2.partido = s.partido
      AND t2.region = s.region
      AND t2.is_current = TRUE
  );

In [0]:
%sql
-- Paso 4: Verificar - palermo debe tener 2 versiones, barrio-nuevo 1
SELECT partido, region, ciudad, valid_from, valid_to, is_current
FROM bootcamp.gold.dim_zona_scd2
WHERE partido IN ('palermo', 'barrio-nuevo')
ORDER BY partido, valid_from;

In [0]:
%sql
-- DDL
CREATE TABLE IF NOT EXISTS bootcamp.gold.dim_zona (
  zona_id    BIGINT GENERATED ALWAYS AS IDENTITY,
  partido    STRING NOT NULL,
  region     STRING NOT NULL,
  ciudad     STRING,
  provincia  STRING DEFAULT 'Buenos Aires',
  pais       STRING DEFAULT 'Argentina',
  _created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP(),
  PRIMARY KEY (zona_id)
)
USING DELTA
COMMENT 'Dimension de zonas geograficas -- BK: partido + region'
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

In [0]:
%sql
-- ETL
INSERT INTO bootcamp.gold.dim_zona (partido, region, ciudad)
SELECT DISTINCT partido, region,
  CASE WHEN region = 'capital federal' THEN 'CABA' ELSE 'GBA' END
FROM bootcamp.silver.propiedades
WHERE partido IS NOT NULL;

In [0]:
%sql
-- Verificar (~36 zonas esperadas)
SELECT COUNT(*) AS total_zonas FROM bootcamp.gold.dim_zona;

In [0]:
%sql
-- dim_tipo_operacion DDL
CREATE TABLE IF NOT EXISTS bootcamp.gold.dim_tipo_operacion (
  tipo_operacion_id  BIGINT GENERATED ALWAYS AS IDENTITY,
  tipo_operacion     STRING NOT NULL,
  categoria          STRING,
  _created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP(),
  PRIMARY KEY (tipo_operacion_id)
)
USING DELTA
COMMENT 'Dimension de tipos de operacion'
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');



In [0]:
%sql
-- dim_tipo_operacion ETL
INSERT INTO bootcamp.gold.dim_tipo_operacion (tipo_operacion, categoria)
SELECT DISTINCT
  tipo_operacion,
  CASE
    WHEN tipo_operacion LIKE '%temporal%' THEN 'temporal'
    WHEN tipo_operacion IN ('alquiler', 'venta') THEN 'residencial'
    ELSE 'otro'
  END AS categoria
FROM bootcamp.silver.propiedades
WHERE tipo_operacion IS NOT NULL;

In [0]:
%sql
-- dim_caracteristicas DDL
CREATE TABLE IF NOT EXISTS bootcamp.gold.dim_caracteristicas (
  caracteristicas_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
  estado             STRING NOT NULL,
  cochera            BOOLEAN,
  _created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING delta
COMMENT 'Dimension de caracteristicas de la propiedad (estado + cochera) - Star Schema'
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

In [0]:
%sql
-- dim_caracteristicas ETL
INSERT INTO bootcamp.gold.dim_caracteristicas (estado, cochera)
SELECT DISTINCT
  COALESCE(estado, 'sin especificar') AS estado,
  COALESCE(cochera, false) AS cochera
FROM bootcamp.silver.propiedades;

In [0]:
%sql
-- Verificar conteos
SELECT 'dim_tipo_operacion' AS dim, COUNT(*) AS total FROM bootcamp.gold.dim_tipo_operacion
UNION ALL
SELECT 'dim_caracteristicas', COUNT(*) FROM bootcamp.gold.dim_caracteristicas;

In [0]:
%sql
-- DDL
CREATE TABLE IF NOT EXISTS bootcamp.gold.dim_tiempo (
  fecha_id          BIGINT NOT NULL,
  fecha             DATE,
  anio              INT,
  mes               INT,
  trimestre         INT,
  dia_semana        STRING,
  es_fin_de_semana  BOOLEAN,
  _created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP(),
  PRIMARY KEY (fecha_id)
)
USING DELTA
COMMENT 'Dimension de tiempo -- fecha_id formato YYYYMMDD'
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

In [0]:
%sql
-- ETL desde fechas de _processing_timestamp
INSERT INTO bootcamp.gold.dim_tiempo (fecha_id, fecha, anio, mes, trimestre, dia_semana, es_fin_de_semana)
SELECT DISTINCT
  CAST(DATE_FORMAT(CAST(_processing_timestamp AS DATE), 'yyyyMMdd') AS BIGINT) AS fecha_id,
  CAST(_processing_timestamp AS DATE) AS fecha,
  YEAR(_processing_timestamp) AS anio,
  MONTH(_processing_timestamp) AS mes,
  QUARTER(_processing_timestamp) AS trimestre,
  DATE_FORMAT(_processing_timestamp, 'EEEE') AS dia_semana,
  CASE WHEN DAYOFWEEK(_processing_timestamp) IN (1, 7) THEN TRUE ELSE FALSE END AS es_fin_de_semana
FROM bootcamp.silver.propiedades
WHERE _processing_timestamp IS NOT NULL;

In [0]:
%sql
-- Verificar
SELECT COUNT(*) AS total_fechas, MIN(fecha) AS fecha_min, MAX(fecha) AS fecha_max
FROM bootcamp.gold.dim_tiempo;

In [0]:
%sql
SELECT 'dim_zona' AS dim_nombre, COUNT(*) AS total_registros, COUNT(*) - COUNT(partido) AS nulls_bk
FROM bootcamp.gold.dim_zona
UNION ALL
SELECT 'dim_tipo_operacion', COUNT(*), COUNT(*) - COUNT(tipo_operacion)
FROM bootcamp.gold.dim_tipo_operacion
UNION ALL
SELECT 'dim_caracteristicas', COUNT(*), COUNT(*) - COUNT(estado)
FROM bootcamp.gold.dim_caracteristicas
UNION ALL
SELECT 'dim_orientacion', COUNT(*), COUNT(*) - COUNT(orientacion)
FROM bootcamp.gold.dim_orientacion
UNION ALL
SELECT 'dim_tiempo', COUNT(*), COUNT(*) - COUNT(fecha)
FROM bootcamp.gold.dim_tiempo;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS bootcamp.gold.fact_propiedades (
  row_hash                    STRING NOT NULL,
  zona_id                     BIGINT,
  tipo_operacion_id           BIGINT,
  fecha_id                    BIGINT,
  caracteristicas_id          BIGINT,
  orientacion_id              BIGINT,
  precio                      DECIMAL(15,2),
  expensas                    DECIMAL(15,2),
  precio_por_m2               DECIMAL(15,2),
  metros_cuadrados_totales    DECIMAL(15,2),
  metros_cuadrados_cubiertos  DECIMAL(15,2),
  ambientes                   INT,
  url                         STRING,
  _refresh_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING delta
COMMENT 'Tabla de hechos de propiedades - Star Schema. PK = row_hash (MD5 de url + precio)'
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

In [0]:
%sql
INSERT INTO bootcamp.gold.fact_propiedades (
  row_hash, zona_id, tipo_operacion_id, fecha_id,
  caracteristicas_id, orientacion_id,
  precio, expensas, precio_por_m2,
  metros_cuadrados_totales, metros_cuadrados_cubiertos,
  ambientes, url
)
SELECT
  MD5(CONCAT_WS('|', sp.url, CAST(sp.precio AS STRING))) AS row_hash,
  dz.zona_id,
  dt.tipo_operacion_id,
  CAST(DATE_FORMAT(sp.fecha_publicacion, 'yyyyMMdd') AS BIGINT) AS fecha_id,
  dc.caracteristicas_id,
  do.orientacion_id,
  sp.precio,
  sp.expensas,
  sp.precio_por_m2,
  sp.metros_cuadrados_totales,
  sp.metros_cuadrados_cubiertos,
  sp.ambientes,
  sp.url
FROM bootcamp.silver.propiedades sp
LEFT JOIN bootcamp.gold.dim_zona dz
  ON sp.partido = dz.partido AND sp.region = dz.region
LEFT JOIN bootcamp.gold.dim_tipo_operacion dt
  ON sp.tipo_operacion = dt.tipo_operacion 
LEFT JOIN bootcamp.gold.dim_caracteristicas dc
  ON COALESCE(sp.estado, 'sin especificar') = dc.estado
  AND COALESCE(sp.cochera, false) = dc.cochera
LEFT JOIN bootcamp.gold.dim_orientacion do
  ON COALESCE(sp.orientacion, 'sin especificar') = do.orientacion
LEFT JOIN bootcamp.gold.dim_tiempo dtm
  ON sp.fecha_publicacion = dtm.fecha;

In [0]:
%sql
-- Verificar: total debe coincidir con Silver
SELECT
  (SELECT COUNT(*) FROM bootcamp.silver.propiedades) AS silver_count,
  (SELECT COUNT(*) FROM bootcamp.gold.fact_propiedades) AS fact_count;

CREANDO DIMENSIONES


In [0]:
-- dim_zona — DDL + ETL

SELECT * FROM bootcamp.gold.dim_zona


In [0]:
SELECT * FROM bootcamp.gold.dim_tipo_operacion

In [0]:
SELECT * FROM bootcamp.gold.dim_tiempo

In [0]:
%sql
CREATE OR REPLACE TABLE bootcamp.gold.fact_propiedades (
  row_hash                    STRING NOT NULL,
  zona_id                     BIGINT,
  tipo_operacion_id           BIGINT,
  fecha_id                    BIGINT,
  caracteristicas_id          BIGINT,
  orientacion_id              BIGINT,
  precio                      DECIMAL(15,2),
  moneda                      STRING NOT NULL,
  expensas                    DECIMAL(15,2),
  precio_por_m2               DECIMAL(15,2),
  metros_cuadrados_totales    DECIMAL(15,2),
  metros_cuadrados_cubiertos  DECIMAL(15,2),
  ambientes                   INT,
  url                         STRING,
  _refresh_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING delta
COMMENT 'Tabla de hechos de propiedades - Star Schema. PK = row_hash (MD5 de url + precio)'
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

In [0]:
truncate table bootcamp.gold.fact_propiedades

In [0]:
%sql
INSERT INTO bootcamp.gold.fact_propiedades (
  row_hash, zona_id, tipo_operacion_id, fecha_id,
  caracteristicas_id, orientacion_id,
  precio, moneda, expensas, precio_por_m2,
  metros_cuadrados_totales, metros_cuadrados_cubiertos,
  ambientes, url
)
WITH silver_dedup AS (
  SELECT row_hash, fecha_publicacion, partido, region, tipo_operacion, estado, cochera,
         orientacion, precio, moneda, expensas, precio_por_m2,
         metros_cuadrados_totales, metros_cuadrados_cubiertos, ambientes, url
  FROM (
    SELECT sp.*,
           MD5(CONCAT_WS('|', sp.url, CAST(sp.precio AS STRING))) AS row_hash,
           ROW_NUMBER() OVER (
             PARTITION BY MD5(CONCAT_WS('|', sp.url, CAST(sp.precio AS STRING)))
             ORDER BY sp._processing_timestamp DESC
           ) AS rn
    FROM bootcamp.silver.propiedades sp
  )
  WHERE rn = 1
)
SELECT
  s.row_hash,
  dz.zona_id,
  dt.tipo_operacion_id,
  CAST(DATE_FORMAT(s.fecha_publicacion, 'yyyyMMdd') AS BIGINT) AS fecha_id,
  dc.caracteristicas_id,
  do.orientacion_id,
  s.precio,
  s.moneda,
  s.expensas,
  s.precio_por_m2,
  s.metros_cuadrados_totales,
  s.metros_cuadrados_cubiertos,
  s.ambientes,
  s.url
FROM silver_dedup s
-- Joins contra dimensiones deduplicadas por clave de negocio
-- (evita fan-out si los ETL de dimensiones se ejecutaron mas de una vez)
LEFT JOIN (
  SELECT partido, region, MIN(zona_id) AS zona_id
  FROM bootcamp.gold.dim_zona
  GROUP BY partido, region
) dz ON s.partido = dz.partido AND s.region = dz.region
LEFT JOIN (
  SELECT tipo_operacion, MIN(tipo_operacion_id) AS tipo_operacion_id
  FROM bootcamp.gold.dim_tipo_operacion
  GROUP BY tipo_operacion
) dt ON s.tipo_operacion = dt.tipo_operacion
LEFT JOIN (
  SELECT estado, cochera, MIN(caracteristicas_id) AS caracteristicas_id
  FROM bootcamp.gold.dim_caracteristicas
  GROUP BY estado, cochera
) dc ON COALESCE(s.estado, 'sin especificar') = dc.estado
  AND COALESCE(s.cochera, false) = dc.cochera
LEFT JOIN (
  SELECT orientacion, MIN(orientacion_id) AS orientacion_id
  FROM bootcamp.gold.dim_orientacion
  GROUP BY orientacion
) do ON COALESCE(s.orientacion, 'sin especificar') = do.orientacion
-- Insert idempotente: no reinserta row_hashes ya cargados
WHERE NOT EXISTS (
  SELECT 1 FROM bootcamp.gold.fact_propiedades f WHERE f.row_hash = s.row_hash
);

In [0]:
%sql
-- Verificar: total debe coincidir con Silver
SELECT
  (SELECT COUNT(*) FROM bootcamp.silver.propiedades) AS silver_count,
  (SELECT COUNT(*) FROM bootcamp.gold.fact_propiedades) AS fact_count;

In [0]:
%sql
SELECT
  COUNT(*) AS total,
  COUNT(zona_id) AS con_zona_id,
  COUNT(*) - COUNT(zona_id) AS sin_zona_id,
  COUNT(tipo_operacion_id) AS con_tipo_op_id,
  COUNT(*) - COUNT(tipo_operacion_id) AS sin_tipo_op_id,
  COUNT(caracteristicas_id) AS con_caract_id,
  COUNT(*) - COUNT(caracteristicas_id) AS sin_caract_id,
  COUNT(orientacion_id) AS con_orient_id,
  COUNT(*) - COUNT(orientacion_id) AS sin_orient_id,
  COUNT(fecha_id) AS con_fecha_id,
  COUNT(*) - COUNT(fecha_id) AS sin_fecha_id
FROM bootcamp.gold.fact_propiedades;